# 01 — Scraping
Downloads all PDF and web sources defined in `data/sources.json` and saves
the scraped corpus locally as `data/scraped_corpus.json`.

**Only needs to be run once** (or when you add new sources to `data/sources.json`).
The output file is git-ignored — see `data/.gitignore`.

**Pre-requisites**
```
pip install -r requirements.txt
cp .env.example .env   # fill in OPENAI_API_KEY
```

In [1]:
# ── 1. Environment setup ─────────────────────────────────────────────────────
import sys
from pathlib import Path

ROOT = Path.cwd().parent  # notebooks/ -> project root
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import config

print("Project root          :", ROOT)
print("Sources manifest      :", config.SOURCES_PATH)
print("Corpus output path    :", config.CORPUS_PATH)
print("PDF download directory:", config.DOWNLOADS_DIR)

Project root          : C:\Users\wenqi\Desktop\talkingtoai
Sources manifest      : C:\Users\wenqi\Desktop\talkingtoai\data\sources.json
Corpus output path    : C:\Users\wenqi\Desktop\talkingtoai\data\scraped_corpus.json
PDF download directory: C:\Users\wenqi\Desktop\talkingtoai\data\downloads


In [2]:
# ── 2. Imports & setup ────────────────────────────────────────────────────────
import httpx
import pdfplumber
import trafilatura
import json
import hashlib
import time
import re
import os
from pathlib import Path
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
from datetime import datetime
from tqdm import tqdm

import config

# Create local working directories
Path(config.DOWNLOADS_DIR).mkdir(parents=True, exist_ok=True)
Path(config.CORPUS_PATH).parent.mkdir(parents=True, exist_ok=True)

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/120.0.0.0 Safari/537.36"
}

print("Setup complete.")

Setup complete.


In [3]:
# ── 3. Load sources from data/sources.json ────────────────────────────────────
# Source URLs live in the repo. To add a new source: edit data/sources.json.

with open(config.SOURCES_PATH, encoding="utf-8") as f:
    sources = json.load(f)

PDF_SOURCES   = sources["pdf_sources"]
CRAWL_SOURCES = sources["crawl_sources"]

print(f"Loaded {len(PDF_SOURCES)} PDF sources and {len(CRAWL_SOURCES)} crawl seeds.")

from collections import Counter
print("\nPDFs by source:")
for k, v in Counter(p["source"] for p in PDF_SOURCES).most_common():
    print(f"  {k:<20} {v}")
print("\nCrawl seeds by source:")
for k, v in Counter(p["source"] for p in CRAWL_SOURCES).most_common():
    print(f"  {k:<20} {v}")

Loaded 36 PDF sources and 33 crawl seeds.

PDFs by source:
  HPB                  13
  MOM                  10
  SGH                  3
  MOH                  2
  ILO                  2
  TAFEP                1
  WSH_Council          1
  NCSS                 1
  WHO                  1
  WHO_ILO              1
  CIPD_Mind            1

Crawl seeds by source:
  SGH                  8
  SingHealth           4
  MOM                  3
  HPB                  3
  AWARE                2
  WSH_Council          1
  TAFEP                1
  MOH                  1
  NCSS                 1
  Silver_Ribbon        1
  SOS                  1
  CIPD                 1
  Mind_UK              1
  SHRM                 1
  SingHealth_HealthXchange 1
  HBR                  1
  Great_Place_To_Work  1
  IHI                  1


In [4]:
# ── 4. Helper functions (unchanged from original) ─────────────────────────────

def clean_text(text: str) -> str:
    """Remove excess whitespace and page number artefacts."""
    if not text:
        return ''
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'^[\s\|\-\.]*\d+[\s\|\-\.]*$', '', text, flags=re.MULTILINE)
    return text.strip()


def scrape_pdf(entry: dict) -> dict | None:
    url   = entry['url']
    title = entry['title']
    safe  = re.sub(r'[^\w\s-]', '', title)[:60].strip().replace(' ', '_')
    path  = Path(config.DOWNLOADS_DIR) / f"{entry['source']}_{safe}.pdf"

    try:
        print(f"  ⬇  {title[:60]}")
        resp = httpx.get(url, timeout=45, follow_redirects=True, headers=HEADERS)
        resp.raise_for_status()
        path.write_bytes(resp.content)

        pages = []
        with pdfplumber.open(path) as pdf:
            for page in pdf.pages:
                raw = page.extract_text()
                if raw:
                    pages.append(clean_text(raw))

        full_text = '\n\n'.join(pages)
        if len(full_text) < 100:
            print(f"  ⚠  Very little text — skipping")
            return None

        print(f"  ✅ {len(pages)} pages | {len(full_text):,} chars")
        return {
            **entry,
            'type':         'pdf',
            'full_text':    full_text,
            'page_count':   len(pages),
            'char_count':   len(full_text),
            'scraped_at':   datetime.utcnow().isoformat(),
            'content_hash': hashlib.sha256(full_text.encode()).hexdigest(),
        }
    except httpx.HTTPStatusError as e:
        print(f"  ❌ HTTP {e.response.status_code} — {title}")
        return None
    except Exception as e:
        print(f"  ❌ Error — {title} — {e}")
        return None


def crawl_website(entry: dict) -> list:
    seed_url  = entry['seed_url']
    max_pages = entry.get('max_pages', 20)
    domain    = urlparse(seed_url).netloc
    SKIP_EXT  = {'.pdf', '.zip', '.xlsx', '.docx', '.png', '.jpg', '.gif', '.svg'}

    visited, queue, results = set(), [seed_url], []
    print(f"  🌐 {entry['source']} — {domain} (max {max_pages} pages)")

    with tqdm(total=max_pages, desc=f"    {entry['source']}", leave=False) as pbar:
        while queue and len(visited) < max_pages:
            url = queue.pop(0).split('#')[0]
            if url in visited or any(url.lower().endswith(e) for e in SKIP_EXT):
                continue
            visited.add(url)
            pbar.update(1)

            try:
                resp = httpx.get(url, timeout=15, headers=HEADERS, follow_redirects=True)
                if resp.status_code != 200:
                    continue
                if 'text/html' not in resp.headers.get('content-type', ''):
                    continue

                html = resp.text
                raw  = trafilatura.extract(html, include_links=False,
                                           include_tables=True, no_fallback=False)
                text = clean_text(raw or '')

                if text and len(text) > 200:
                    results.append({
                        **{k: v for k, v in entry.items() if k not in ('seed_url', 'max_pages')},
                        'type':         'webpage',
                        'url':          url,
                        'full_text':    text,
                        'char_count':   len(text),
                        'scraped_at':   datetime.utcnow().isoformat(),
                        'content_hash': hashlib.sha256(text.encode()).hexdigest(),
                    })

                soup = BeautifulSoup(html, 'html.parser')
                for tag in soup.find_all('a', href=True):
                    abs_url = urljoin(url, tag['href'])
                    if urlparse(abs_url).netloc == domain and abs_url not in visited:
                        queue.append(abs_url)

                time.sleep(0.5)
            except Exception:
                continue

    print(f"  ✅ {len(results)} pages scraped from {domain}")
    return results


print('Helper functions ready.')

Helper functions ready.


In [5]:
# ── 5. Run scraping ───────────────────────────────────────────────────────────
all_documents = []
seen_hashes   = set()
failed        = []

# PART A — PDFs
print('=' * 60)
print(f'PART A — PDFs ({len(PDF_SOURCES)} sources)')
print('=' * 60)

for entry in tqdm(PDF_SOURCES, desc='All PDFs'):
    doc = scrape_pdf(entry)
    if doc and doc['content_hash'] not in seen_hashes:
        all_documents.append(doc)
        seen_hashes.add(doc['content_hash'])
    elif not doc:
        failed.append({'type': 'pdf', 'title': entry['title'], 'url': entry['url']})

print(f"\nPDFs: {sum(1 for d in all_documents if d['type']=='pdf')} ok, "
      f"{sum(1 for f in failed if f['type']=='pdf')} failed")

# PART B — Web crawls
print('\n' + '=' * 60)
print(f'PART B — Web crawls ({len(CRAWL_SOURCES)} seeds)')
print('=' * 60)

for i, entry in enumerate(CRAWL_SOURCES, 1):
    print(f"\n[{i}/{len(CRAWL_SOURCES)}] {entry['source']} ({entry['region']})")
    pages = crawl_website(entry)
    added = 0
    for page in pages:
        if page['content_hash'] not in seen_hashes:
            all_documents.append(page)
            seen_hashes.add(page['content_hash'])
            added += 1
    if added == 0:
        failed.append({'type': 'crawl', 'source': entry['source'], 'url': entry['seed_url']})

# PART C — Save
output_path = config.CORPUS_PATH
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(all_documents, f, ensure_ascii=False, indent=2)

print('\n' + '=' * 60)
print('DONE')
print('=' * 60)
print(f'  Documents : {len(all_documents)}')
print(f'  Total chars: {sum(d["char_count"] for d in all_documents):,}')
print(f'  Saved to  : {output_path}')
if failed:
    print(f'\n  ⚠ {len(failed)} sources failed:')
    for f in failed:
        print(f"    - {f.get('title', f.get('source'))}")

PART A — PDFs (36 sources)


All PDFs:   0%|                                                                                                             | 0/36 [00:00<?, ?it/s]

  ⬇  Tripartite Advisory on Mental Health and Well-Being at Workp


All PDFs:   3%|██▊                                                                                                  | 1/36 [00:03<01:52,  3.22s/it]

  ✅ 26 pages | 31,458 chars
  ⬇  Media Fact Sheet: Workplace Mental Well-being Strategies 202


...

  ✅ 20 pages scraped from www.healthhub.sg

[30/33] SingHealth_HealthXchange (singapore)
  🌐 SingHealth_HealthXchange — www.healthxchange.sg (max 1 pages)


  ✅ 1 pages scraped from www.healthxchange.sg

[31/33] HBR (international)
  🌐 HBR — hbr.org (max 1 pages)


  ✅ 1 pages scraped from hbr.org

[32/33] Great_Place_To_Work (international)
  🌐 Great_Place_To_Work — www.greatplacetowork.com (max 1 pages)


  ✅ 1 pages scraped from www.greatplacetowork.com

[33/33] IHI (international)
  🌐 IHI — www.ihi.org (max 5 pages)


  ✅ 5 pages scraped from www.ihi.org

DONE
  Documents : 278
  Total chars: 2,100,680
  Saved to  : C:\Users\wenqi\Desktop\talkingtoai\data\scraped_corpus.json

  ⚠ 32 sources failed:
    - HPB WHP Toolkit 1: Understanding Workplace Health Promotion
    - HPB WHP Toolkit 2: Benchmarking WHP Programme
    - HPB WHP Toolkit 3: Securing Management Support
    - HPB WHP Toolkit 4: Setting Up Committee Structures
    - HPB WHP Toolkit 5a: Uncovering True Health Needs
    - HPB WHP Toolkit 5b: Choosing Interventions That Work
    - HPB WHP Toolkit 5c: Writing a Programme Plan
    - HPB WHP Toolkit 6: Maximising Participation
    - HPB WHP Toolkit 7: Programme Implementation FAQs
    - HPB WHP Toolkit 8: Programme Evaluation
    - Workplace Alliance for Health WAH Scheme Funding Guidelines
    - National Population Health Survey 2024
    - WHO Guidelines on Mental Health at Work 2022
    - Mental Health at Work WHO ILO Policy Brief 2022
    - People Managers Guide to Mental Health at Work CIP

In [6]:
# ── 6. Verify corpus ──────────────────────────────────────────────────────────
from collections import Counter

print(f'Total documents : {len(all_documents)}')
print(f'Total chars     : {sum(d["char_count"] for d in all_documents):,}')

def print_table(title, counter):
    print(f'\n{title}')
    for k, v in counter.most_common():
        print(f'  {k:<28} {v:>4} docs')

print_table('By source:',   Counter(d['source']   for d in all_documents))
print_table('By category:', Counter(d['category'] for d in all_documents))
print_table('By type:',     Counter(d['type']     for d in all_documents))

s = all_documents[0]
print(f'\nFirst doc preview:')
print(f'  {s.get("title", s.get("url"))} [{s["source"]}]')
print(s['full_text'][:400])

Total documents : 278
Total chars     : 2,100,680

By source:
  HPB                            50 docs
  CIPD                           39 docs
  MOM                            36 docs
  NCSS                           26 docs
  SHRM                           26 docs
  WSH_Council                    21 docs
  AWARE                          20 docs
  SOS                            19 docs
  MOH                            16 docs
  TAFEP                          15 docs
  IHI                             4 docs
  ILO                             2 docs
  SGH                             1 docs
  SingHealth_HealthXchange        1 docs
  HBR                             1 docs
  Great_Place_To_Work             1 docs

By category:
  mental_health                 136 docs
  workplace_wellness             31 docs
  physical_wellness              22 docs
  harassment                     21 docs
  policy                         20 docs
  crisis_support                 19 docs
  workplace_fairness  

In [7]:
# ── 7. (Optional) Inspect a sample document ──────────────────────────────────
# The corpus is saved locally. Open it in any JSON viewer, or preview here:
import json
with open(config.CORPUS_PATH, encoding="utf-8") as f:
    sample = json.load(f)
doc = sample[0]
print("Source:", doc.get("source"), "|", doc.get("title"))
print("Chars :", doc.get("char_count"))
print("Preview:")
print(doc.get("full_text", "")[:500])

Source: MOM | Tripartite Advisory on Mental Health and Well-Being at Workplaces
Chars : 31458
Preview:
TRIPARTITE ADVISORY ON
MENTAL HEALTH AND WELL-
BEING AT WORKPLACES
A Tripartite Advisory jointly issued by
Ministry of Manpower (MOM),
National Trades Union Congress (NTUC) and
Singapore National Employers Federation (SNEF)
First edition: 17 Nov 2020
Second edition: 20 Nov 2023

PURPOSE
Mental health is a growing concern. The 2022
National Population Health Survey found that
prevalence of poor mental health among Singapore
residents aged 18 to 74 was 17.0%. International
studies have suggested t
